# Etna Cause–Trigger Analysis

This notebook applies the current Cause–Trigger implementation to the fixed Etna case-study interval around the Wenchuan teleseismic arrival.

The analysis is organized to avoid post-hoc hyperparameter selection:

- the case interval, effect, minimum interval lengths, and maximum lag are fixed before fitting;
- VAR-AIC and VAR-BIC are lag references only;
- the primary result is stability across lags and causal-discovery backends;
- PCMCI+ contemporaneous-trigger testing is retained as an explicitly labelled extension;
- the known Wenchuan time is contextual and does not determine the algorithmic split.

Two aligned standardized frames are used:

- `X_model`: case-standardized data for causal discovery, coefficient estimation, \(V\), and moderation;
- `X_mean`: pre-case-reference-standardized data for the split and mean-comparison steps.

Accepted pairs are displayed as **Cause | Trigger**. For Etna, the target effect is anomalous local catalogue seismicity. The exported audit files preserve every backend–lag result and every accepted or rejected moderation test, while the notebook displays only the stability summaries needed for interpretation.


## Imports and configuration


In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cause_trigger_cases import ETNA_CASE
from cause_trigger_reporting import (
    backend_stability_summary,
    delayed_lag_correlation,
    export_audit_csvs,
    pair_stability_summary,
    plot_effect_with_split,
    reference_moderation_table,
    run_complete_grid,
    show_table,
)
from cause_trigger_workflow import (
    COMPACT_RUN_SPECS,
    WorkflowConfig,
    case_study_interval,
    load_model_frame,
    pre_case_reference_interval,
    prepare_case_frames,
    reference_parameter_table,
    run_one,
    split_diagnostics,
)

CASE = ETNA_CASE
EFFECT = CASE.effect
MODEL_COLUMNS = CASE.model_columns
VARIABLE_LABELS = CASE.variable_labels

DATA_PATH = PROJECT_ROOT / "data" / "etna" / "etna_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

SAVE_RESULTS = True
SHOW_FULL_GRID = False
SHOW_SCALING_REPORT = False

EVENT_TIME = pd.Timestamp("2008-05-12 06:28:00", tz="UTC")

CASE_PRE_DAYS = 3
CASE_POST_HOURS = 72
MIN_I1_LENGTH = 48
MIN_I2_LENGTH = 48

MAX_LAGS = 12
LAG_GRID = tuple(range(1, MAX_LAGS + 1))

REFERENCE_DAYS = 14
REFERENCE_MIN_COVERAGE = 0.90

ALPHA = 0.05
PCMCI_PC_ALPHA = 0.05
PCMCI_ALPHA_LEVEL = 0.05
PCMCI_FDR_METHOD = "fdr_bh"
COND_IND_TEST = "robust_parcorr"


### Fixed data and split

The primary interval contains **3 days before** and 72 hours after the Wenchuan arrival. Both \(I_1\) and \(I_2\) must contain at least 48 hourly observations. Lags from 1 to 12 hours are evaluated.

The CSV remains unstandardized; transformations and scaling are fitted within this workflow. The known Wenchuan time is plotted for context but does not constrain the split.


In [ ]:
X_full_raw = load_model_frame(
    DATA_PATH,
    include_columns=MODEL_COLUMNS,
    require_complete=False,
    index_name=CASE.index_name,
)

X_case_raw = case_study_interval(
    X_full_raw,
    EVENT_TIME,
    pre_days=CASE_PRE_DAYS,
    post_hours=CASE_POST_HOURS,
)

reference_raw = pre_case_reference_interval(
    X_full_raw,
    case_start=X_case_raw.index.min(),
    reference_days=REFERENCE_DAYS,
    min_coverage=REFERENCE_MIN_COVERAGE,
    case_name=CASE.name,
)

X_model, X_mean, scaling_report = prepare_case_frames(
    X_case_raw,
    reference_raw,
    case=CASE,
)

reference_parameters = reference_parameter_table(
    X_model,
    EFFECT,
    fallback_lag=1,
)
REFERENCE_LAGS = tuple(
    int(value)
    for value in sorted(
        reference_parameters["selected_lag"].astype(int).unique()
    )
)

workflow = WorkflowConfig(
    effect=EFFECT,
    alpha=ALPHA,
    selected_lag=REFERENCE_LAGS[0],
    min_I1_length=MIN_I1_LENGTH,
    min_I2_length=MIN_I2_LENGTH,
    distribution="gaussian",
    refit_alpha=1.0,
    refit_cv=True,
    refit_cv_folds=3,
    pcmci_pc_alpha=PCMCI_PC_ALPHA,
    pcmci_alpha_level=PCMCI_ALPHA_LEVEL,
    pcmci_fdr_method=PCMCI_FDR_METHOD,
    pcmci_cond_ind_test=COND_IND_TEST,
    pcmci_plus_use_contemporaneous_triggers=False,
)

split_row = split_diagnostics(
    X_mean,
    EFFECT,
    event_time=EVENT_TIME,
    min_I1_length=MIN_I1_LENGTH,
    min_I2_length=MIN_I2_LENGTH,
)

lag_reference_text = "; ".join(
    f"{row.criterion}={int(row.selected_lag)} h"
    for row in reference_parameters.itertuples()
)

effect_peak = X_case_raw[CASE.raw_effect].idxmax()

design_summary = pd.DataFrame([
    {
        "Item": "Case interval",
        "Value": (
            f"{X_case_raw.index.min()} to {X_case_raw.index.max()} "
            f"({len(X_case_raw)} h)"
        ),
    },
    {
        "Item": "Reference coverage",
        "Value": (
            f"{len(reference_raw)}/"
            f"{reference_raw.attrs['expected_rows']} h "
            f"({reference_raw.attrs['coverage']:.1%})"
        ),
    },

    {"Item": "Wenchuan arrival", "Value": str(EVENT_TIME)},
    {
        "Item": "Effect maximum",
        "Value": f"{effect_peak} ({effect_peak - EVENT_TIME})",
    },

    {
        "Item": "Automatic split",
        "Value": (
            f"{split_row['split_time']}; "
            f"I1={split_row['I1_length']} h, "
            f"I2={split_row['I2_length']} h"
        ),
    },
    {
        "Item": "Split status",
        "Value": (
            "Boundary solution; "
            f"{split_row['distance_to_event']} after Wenchuan"
),
    },
    {"Item": "VAR lag references", "Value": lag_reference_text},
])

show_table(design_summary, "Fixed design and timing")

if SHOW_SCALING_REPORT:
    show_table(
        scaling_report.reset_index(names="Variable"),
        "Transformation and scaling audit",
    )

plot_effect_with_split(
    X_mean,
    EFFECT,
    split_row,
    event_time=EVENT_TIME,
    event_label="Wenchuan earthquake",
    title=None,
    filename="etna_effect_split",
    save_dir=FIGURES_DIR,
    formats=("pdf", "png"),
)


## Primary backend-by-lag stability analysis

Each backend is run **once at every lag from 1 to 12 hours** under the same preprocessing, split rule, and significance level:

- **HMML**: paper-compatible primary causal-discovery backend;
- **PCMCI**: lagged causal discovery;
- **PCMCI+ \(tau=0\) extension**: lagged discovery plus eligible directed contemporaneous links as trigger candidates.

The notebook displays compact stability summaries. The complete backend–lag grid and every moderation diagnostic are saved to CSV, including no-signal runs and rejected candidate pairs.


In [ ]:
lag_grid, all_diagnostics = run_complete_grid(
    X_model,
    X_mean,
    workflow,
    run_one=run_one,
    run_specs=COMPACT_RUN_SPECS,
    lags=LAG_GRID,
    cond_ind_test=COND_IND_TEST,
)

errors = lag_grid.loc[lag_grid["error"].notna()].copy()
backend_summary = backend_stability_summary(
    lag_grid,
    lag_count=len(LAG_GRID),
)
pair_summary = pair_stability_summary(
    lag_grid,
    lag_count=len(LAG_GRID),
    variable_labels=VARIABLE_LABELS,
)

show_table(backend_summary, "Backend stability")
show_table(pair_summary, "Accepted cause–trigger pairs across lags")

if not pair_summary.empty:
    top = pair_summary.iloc[0]
    print(
        "Most recurrent accepted pair: "
        f"{top['Cause']} | {top['Trigger']} "
        f"at {top['Lag support']} lags."
    )

if not errors.empty:
    show_table(
        errors[["run", "backend", "lag", "error"]],
        "Execution errors",
    )

if SHOW_FULL_GRID:
    show_table(
        lag_grid.drop(columns="error"),
        "Complete backend-by-lag grid",
    )


## VAR reference-lag check

In [ ]:
reference_display = reference_moderation_table(
    all_diagnostics,
    reference_parameters,
    variable_labels=VARIABLE_LABELS,
)

if reference_display.empty:
    print(
        "No cause–trigger pair passed moderation at the "
        f"AIC/BIC reference lags {REFERENCE_LAGS}."
    )
else:
    show_table(
        reference_display,
        "Accepted pairs at VAR reference lags",
    )


## Etna-specific delayed-lag diagnostic

The catalogue-response episode begins well after the main Wenchuan pulse. This descriptive scan evaluates the correlation between the transformed teleseismic proxy at \(t-tau\) and the effect at \(t\) over the complete case interval for delays up to 30 hours.

It does **not** replace the Cause–Trigger analysis, estimate a causal delay, or justify selecting the lag with the largest correlation.


In [ ]:
MAX_DELAY_HOURS = 30

delayed_lag_scan = delayed_lag_correlation(
    X_model,
    effect=EFFECT,
    predictor="teleseismic_scaled",
    max_lag=MAX_DELAY_HOURS,
)

top_delays = (
    delayed_lag_scan
    .assign(abs_correlation=lambda frame: frame["correlation"].abs())
    .sort_values("abs_correlation", ascending=False)
    .drop(columns="abs_correlation")
    .head(10)
)

show_table(
    top_delays.rename(columns={
        "lag_hours": "Delay (h)",
        "correlation": "Correlation",
        "n_aligned": "Aligned rows",
    }),
    "Largest absolute full-case teleseismic–effect correlations",
)


## Saved audit outputs

The notebook writes exactly three CSV files:

1. **All runs** — one row for every backend × lag combination, including empty results and errors.
2. **Moderation diagnostics** — every tested cause–trigger pair, accepted or rejected, with the complete test statistics and reason.
3. **Analysis summary** — design, split, lag references, backend stability, pair stability, and the final interpretation summary.

Nested lists and pair structures are JSON-encoded inside CSV cells so they can be read reliably in Python, R, or a spreadsheet.


In [ ]:
if SAVE_RESULTS:
    saved_files = export_audit_csvs(
        results_dir=RESULTS_DIR,
        case_prefix="etna",
        lag_grid=lag_grid,
        diagnostics=all_diagnostics,
        design=design_summary,
        lag_references=reference_parameters,
        backend_summary=backend_summary,
        pair_summary=pair_summary,
        conclusion=(
            "Exploratory HMML-specific cause–trigger candidate; "
            "no backend-consistent identification of Wenchuan as a "
            "trigger under the current split-based architecture."
        ),
        variable_labels=VARIABLE_LABELS,
        delayed_scan=delayed_lag_scan,
    )
    show_table(saved_files, "Saved result files")
    print(f"Results directory: {RESULTS_DIR}")
else:
    print("Result export disabled: set SAVE_RESULTS = True.")


## Interpretation

The **most recurrent accepted result** is HMML's classification of **past local seismicity as the cause and teleseismic RMS as the trigger**, occurring at 7 of the 12 tested lags. This is meaningful within HMML, but it is not backend-consistent: PCMCI produced no accepted pair, and PCMCI+ produced only one isolated result with the opposite orientation. The AIC and BIC reference lags also produced no accepted pair.

The split is a boundary solution at 2008-05-13 06:00 UTC, almost 24 hours after Wenchuan. Because causal discovery and moderation on \(I_2\) use values contained within \(I_2\), the recurrent HMML pair does **not directly test the main Wenchuan pulse**, which lies in \(I_1\). It should therefore be described as a post-split association involving the teleseismic-band proxy, not as definitive detection of Wenchuan as the trigger.

The full-case descriptive scan peaks near 28–30 hours, consistent with a delayed relation between the teleseismic arrival and the catalogue-response episode. This supports the timing argument but is not a causal-lag estimate.

**Conclusion:** Etna provides an exploratory HMML-specific cause–trigger candidate, but no backend-consistent identification of Wenchuan as a trigger under the present split-based architecture.
